# Xcapit FHE-ML Platform - Retail: Prediccion de Churn de Clientes

## Caso de Uso: Consorcio de Retailers LATAM

Este notebook demuestra el flujo completo de:
1. Registro y configuracion del consorcio de retailers
2. Generacion de datos sinteticos de clientes
3. Encriptacion con FHE (CKKS)
4. Votacion commit-reveal para gobernanza
5. Entrenamiento de modelo de prediccion de churn
6. Predicciones y estrategias de retencion

### Escenario
Tres cadenas de retail de Argentina quieren colaborar para predecir y prevenir la fuga de clientes sin compartir datos sensibles de compras.

## 1. Setup e Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import numpy as np
import pandas as pd
import hashlib
import secrets
from datetime import datetime, timedelta
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("Xcapit FHE-ML Platform - Retail Demo")
print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)

## 2. Configuracion del Consorcio de Retailers

In [ ]:
# Configuracion del dataset de retail
RETAIL_CONFIG = {
    "n_samples": 10000,
    "churn_rate": 0.18,  # 18% de churn
    "retailers": [
        {"name": "Retail Norte", "region": "Buenos Aires", "samples": 4000},
        {"name": "Retail Centro", "region": "Cordoba", "samples": 3000},
        {"name": "Retail Sur", "region": "Patagonia", "samples": 3000},
    ],
    "features": [
        {"name": "tenure_months", "type": "int", "min": 1, "max": 72},
        {"name": "recency_days", "type": "int", "min": 1, "max": 365},
        {"name": "frequency", "type": "int", "min": 1, "max": 100},
        {"name": "monetary_value", "type": "float", "min": 10, "max": 10000},
        {"name": "avg_order_value", "type": "float", "min": 10, "max": 500},
        {"name": "num_returns", "type": "int", "min": 0, "max": 20},
        {"name": "num_complaints", "type": "int", "min": 0, "max": 10},
        {"name": "channel_preference", "type": "category", "values": ["online", "store", "both"]},
        {"name": "loyalty_program", "type": "bool", "true_ratio": 0.4},
        {"name": "marketing_opt_in", "type": "bool", "true_ratio": 0.6},
    ]
}

print("Configuracion Retail:")
print(f"  Total muestras: {RETAIL_CONFIG['n_samples']:,}")
print(f"  Tasa de churn: {RETAIL_CONFIG['churn_rate']*100}%")
print(f"  Retailers participantes: {len(RETAIL_CONFIG['retailers'])}")
print(f"  Features: {len(RETAIL_CONFIG['features'])}")
print("\nRetailers:")
for retailer in RETAIL_CONFIG['retailers']:
    print(f"   {retailer['name']} ({retailer['region']}): {retailer['samples']:,} clientes")

## 3. Generacion de Datos de Clientes (RFM + Engagement)

In [ ]:
def generate_retail_data(config: dict, seed: int = 42) -> pd.DataFrame:
    """Genera datos sinteticos de clientes retail con metricas RFM."""
    np.random.seed(seed)
    
    n = config["n_samples"]
    
    # Generar features basicas con sklearn para correlaciones
    X, y = make_classification(
        n_samples=n,
        n_features=10,
        n_informative=7,
        n_redundant=2,
        n_classes=2,
        weights=[1 - config["churn_rate"], config["churn_rate"]],
        random_state=seed
    )
    
    df = pd.DataFrame()
    
    # Tenure: 1-72 months (clientes nuevos mas propensos a churn)
    df['tenure_months'] = np.abs(X[:, 0]) * 20 + 6
    df.loc[y == 1, 'tenure_months'] = np.random.exponential(8, y.sum()) + 1
    df['tenure_months'] = df['tenure_months'].clip(1, 72).astype(int)
    
    # Recency: 1-365 days (mas dias = mas probable churn)
    df['recency_days'] = np.abs(X[:, 1]) * 60 + 10
    df.loc[y == 1, 'recency_days'] = np.random.exponential(120, y.sum()) + 30
    df['recency_days'] = df['recency_days'].clip(1, 365).astype(int)
    
    # Frequency: 1-100 purchases (menos compras = mas churn)
    df['frequency'] = np.abs(X[:, 2]) * 25 + 10
    df.loc[y == 1, 'frequency'] = np.random.exponential(5, y.sum()) + 1
    df['frequency'] = df['frequency'].clip(1, 100).astype(int)
    
    # Monetary value: $10-$10,000 lifetime value
    df['monetary_value'] = np.abs(X[:, 3]) * 2000 + 200
    df.loc[y == 1, 'monetary_value'] *= 0.4  # Churners gastan menos
    df['monetary_value'] = df['monetary_value'].clip(10, 10000).round(2)
    
    # Avg order value
    df['avg_order_value'] = df['monetary_value'] / df['frequency']
    df['avg_order_value'] = df['avg_order_value'].clip(10, 500).round(2)
    
    # Returns (mas devoluciones = insatisfaccion = churn)
    df['num_returns'] = np.random.poisson(1, n)
    df.loc[y == 1, 'num_returns'] = np.random.poisson(4, y.sum())
    df['num_returns'] = df['num_returns'].clip(0, 20)
    
    # Complaints (mas quejas = mas churn)
    df['num_complaints'] = np.random.poisson(0.3, n)
    df.loc[y == 1, 'num_complaints'] = np.random.poisson(2, y.sum())
    df['num_complaints'] = df['num_complaints'].clip(0, 10)
    
    # Channel preference
    channels = config['features'][7]['values']
    df['channel_preference'] = np.random.choice(channels, n, p=[0.30, 0.40, 0.30])
    
    # Loyalty program (miembros tienen menor churn)
    df['loyalty_program'] = np.random.random(n) < 0.45
    df.loc[y == 1, 'loyalty_program'] = np.random.random(y.sum()) < 0.15
    
    # Marketing opt-in
    df['marketing_opt_in'] = np.random.random(n) < 0.65
    df.loc[y == 1, 'marketing_opt_in'] = np.random.random(y.sum()) < 0.35
    
    # Target
    df['churned'] = y
    
    # Assign to retailers
    retailer_labels = []
    for retailer in config['retailers']:
        retailer_labels.extend([retailer['name']] * retailer['samples'])
    df['retailer'] = retailer_labels[:n]
    
    # Generate customer IDs
    df['customer_id'] = [f"CUS-{hashlib.sha256(str(i).encode()).hexdigest()[:8].upper()}" for i in range(n)]
    
    return df

# Generar datos
df_customers = generate_retail_data(RETAIL_CONFIG)

print("\nDataset Generado:")
print(f"  Shape: {df_customers.shape}")
print(f"  Churned: {df_customers['churned'].sum()} ({df_customers['churned'].mean()*100:.2f}%)")
print(f"\nDistribucion por retailer:")
print(df_customers.groupby('retailer')['churned'].agg(['count', 'sum', 'mean']).round(4))

In [ ]:
# Vista previa de los datos
print("Vista previa de clientes:")
print("="*100)
display_cols = ['customer_id', 'tenure_months', 'recency_days', 'frequency', 'monetary_value',
                'loyalty_program', 'num_complaints', 'churned', 'retailer']
df_customers[display_cols].head(10)

## 4. Analisis RFM y Segmentacion

In [ ]:
print("ANALISIS RFM (Recency, Frequency, Monetary)")
print("=" * 60)

# Calcular scores RFM
df_customers['R_score'] = pd.qcut(df_customers['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
df_customers['F_score'] = pd.qcut(df_customers['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
df_customers['M_score'] = pd.qcut(df_customers['monetary_value'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
df_customers['RFM_score'] = df_customers['R_score'] + df_customers['F_score'] + df_customers['M_score']

# Segmentacion
def segment_customer(rfm):
    if rfm >= 12:
        return 'Champions'
    elif rfm >= 9:
        return 'Loyal'
    elif rfm >= 6:
        return 'At Risk'
    else:
        return 'Lost'

df_customers['segment'] = df_customers['RFM_score'].apply(segment_customer)

print("\nDistribucion de segmentos:")
print("-" * 40)
segment_stats = df_customers.groupby('segment').agg({
    'customer_id': 'count',
    'churned': 'mean',
    'monetary_value': 'mean'
}).round(2)
segment_stats.columns = ['Clientes', 'Tasa Churn', 'Valor Promedio']
segment_stats['Tasa Churn'] = (segment_stats['Tasa Churn'] * 100).round(1).astype(str) + '%'
segment_stats['Valor Promedio'] = '$' + segment_stats['Valor Promedio'].astype(str)
print(segment_stats)

In [ ]:
print("\nCOMPARACION: ACTIVOS vs CHURNED")
print("=" * 60)

metrics = ['tenure_months', 'recency_days', 'frequency', 'monetary_value', 
           'num_returns', 'num_complaints']

comparison = df_customers.groupby('churned')[metrics].mean().round(2).T
comparison.columns = ['Activos', 'Churned']
comparison['Diferencia'] = ((comparison['Churned'] - comparison['Activos']) / comparison['Activos'] * 100).round(1)
print(comparison)

## 5. Encriptacion con FHE (CKKS)

In [ ]:
def simulate_encryption(data: np.ndarray) -> dict:
    """Simula la encriptacion FHE para demo."""
    data_bytes = data.tobytes()
    cipher_hash = hashlib.sha256(data_bytes).hexdigest()
    
    return {
        "ciphertext_preview": f"0x{cipher_hash[:64]}",
        "original_shape": data.shape,
        "encrypted_size_kb": len(data_bytes) * 100 // 1024,
        "scheme": "CKKS",
        "security_bits": 128,
        "poly_modulus_degree": 8192
    }

# Preparar datos para encriptacion
feature_cols = ['tenure_months', 'recency_days', 'frequency', 'monetary_value',
                'avg_order_value', 'num_returns', 'num_complaints']

# Encoding de categoricas
df_encoded = df_customers.copy()
le_channel = LabelEncoder()
df_encoded['channel_enc'] = le_channel.fit_transform(df_encoded['channel_preference'])
df_encoded['loyalty_program'] = df_encoded['loyalty_program'].astype(int)
df_encoded['marketing_opt_in'] = df_encoded['marketing_opt_in'].astype(int)

feature_cols_full = feature_cols + ['channel_enc', 'loyalty_program', 'marketing_opt_in']

X = df_encoded[feature_cols_full].values
y = df_encoded['churned'].values

# Normalizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos preparados para encriptacion:")
print(f"  X shape: {X_scaled.shape}")
print(f"  y shape: {y.shape}")
print(f"  Features: {feature_cols_full}")

In [ ]:
# Demostrar diferencia entre plaintext y ciphertext
print("COMPARACION: PLAINTEXT vs CIPHERTEXT")
print("=" * 60)

# Muestra de datos en plaintext
sample_idx = 0
sample_plaintext = df_customers.iloc[sample_idx]

print("\n[PLAINTEXT] - Datos expuestos:")
print("-" * 40)
print(f"  Customer ID: {sample_plaintext['customer_id']}")
print(f"  Antiguedad: {sample_plaintext['tenure_months']} meses")
print(f"  Ultima compra: hace {sample_plaintext['recency_days']} dias")
print(f"  Frecuencia: {sample_plaintext['frequency']} compras")
print(f"  Valor total: ${sample_plaintext['monetary_value']:,.2f}")
print(f"  Programa fidelidad: {'Si' if sample_plaintext['loyalty_program'] else 'No'}")
print(f"  Quejas: {sample_plaintext['num_complaints']}")
print(f"  Churned: {'SI' if sample_plaintext['churned'] else 'No'}")

# Simular encriptacion
encrypted_info = simulate_encryption(X_scaled[sample_idx:sample_idx+1])

print("\n[CIPHERTEXT] - Datos protegidos:")
print("-" * 40)
print(f"  Datos: {encrypted_info['ciphertext_preview'][:32]}...")
print(f"  Tamano: ~{encrypted_info['encrypted_size_kb']} KB")
print(f"  Esquema: {encrypted_info['scheme']}")
print(f"  Seguridad: {encrypted_info['security_bits']} bits")
print(f"  Poly degree: {encrypted_info['poly_modulus_degree']}")

print("\n" + "=" * 60)
print("Los datos del cliente estan completamente protegidos!")

## 6. Contribuciones de los Retailers (Consorcio)

In [ ]:
print("CONTRIBUCIONES DEL CONSORCIO")
print("=" * 60)

contributions = []
for retailer in RETAIL_CONFIG['retailers']:
    retailer_mask = df_customers['retailer'] == retailer['name']
    retailer_data = df_encoded[retailer_mask][feature_cols_full].values
    retailer_labels = df_customers[retailer_mask]['churned'].values
    
    # Simular hash de contribucion
    data_hash = hashlib.sha256(retailer_data.tobytes()).hexdigest()[:32]
    
    # Estadisticas adicionales
    total_revenue = df_customers[retailer_mask]['monetary_value'].sum()
    avg_tenure = df_customers[retailer_mask]['tenure_months'].mean()
    
    contribution = {
        "retailer": retailer['name'],
        "region": retailer['region'],
        "records": len(retailer_data),
        "churn_count": retailer_labels.sum(),
        "churn_rate": retailer_labels.mean() * 100,
        "total_revenue": total_revenue,
        "avg_tenure": avg_tenure,
        "data_hash": data_hash
    }
    contributions.append(contribution)
    
    print(f"\n{retailer['name']} ({retailer['region']}):")
    print(f"   Clientes: {contribution['records']:,}")
    print(f"   Churned: {contribution['churn_count']} ({contribution['churn_rate']:.2f}%)")
    print(f"   Revenue total: ${contribution['total_revenue']:,.2f}")
    print(f"   Antiguedad promedio: {contribution['avg_tenure']:.1f} meses")
    print(f"   Hash encriptado: {contribution['data_hash']}...")

print("\n" + "=" * 60)
total_records = sum(c['records'] for c in contributions)
total_churn = sum(c['churn_count'] for c in contributions)
total_revenue = sum(c['total_revenue'] for c in contributions)
print(f"Total consorcio: {total_records:,} clientes, {total_churn} churned")
print(f"Revenue total: ${total_revenue:,.2f}")

## 7. Votacion Commit-Reveal (Gobernanza)

In [ ]:
print("VOTACION COMMIT-REVEAL")
print("=" * 60)
print("Propuesta: Entrenar modelo de prediccion de churn")
print("Quorum requerido: 51%")

proposal_id = hashlib.sha256(b"TRAIN_CHURN_MODEL_2025").hexdigest()

# Fase 1: Commit
print("\n[FASE 1: COMMIT] - Votos ocultos")
print("-" * 40)

votes_secret = {}
commitments = {}

for retailer in RETAIL_CONFIG['retailers']:
    vote = True  # Todos votan SI
    salt = secrets.token_bytes(32)
    
    commitment = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    votes_secret[retailer['name']] = (vote, salt)
    commitments[retailer['name']] = commitment
    
    print(f"  {retailer['name']}: 0x{commitment[:24]}... (voto oculto)")

# Fase 2: Reveal
print("\n[FASE 2: REVEAL] - Votos verificados")
print("-" * 40)

yes_votes = 0
for retailer_name, (vote, salt) in votes_secret.items():
    expected = hashlib.sha256(
        proposal_id.encode() + bytes([vote]) + salt
    ).hexdigest()
    
    verified = expected == commitments[retailer_name]
    
    if verified and vote:
        yes_votes += 1
    
    status = "VERIFICADO" if verified else "INVALIDO"
    vote_str = "SI" if vote else "NO"
    print(f"  {retailer_name}: {vote_str} - {status}")

print("\n" + "=" * 60)
approval_pct = (yes_votes / len(RETAIL_CONFIG['retailers'])) * 100
print(f"Resultado: {yes_votes}/{len(RETAIL_CONFIG['retailers'])} ({approval_pct:.0f}%)")
print(f"Estado: {'APROBADO - Entrenamiento autorizado' if approval_pct >= 51 else 'RECHAZADO'}")

## 8. Entrenamiento del Modelo

In [ ]:
print("ENTRENAMIENTO DEL MODELO")
print("=" * 60)

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {len(X_train):,} muestras")
print(f"Datos de prueba: {len(X_test):,} muestras")
print(f"Churn en train: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Churn en test: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

# Entrenar modelo (Gradient Boosting para mejor prediccion de churn)
print("\nEntrenando Gradient Boosting Classifier...")
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metricas
accuracy = accuracy_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_prob)

print("\nEntrenamiento completado!")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"AUC-ROC: {auc_roc:.4f}")

In [ ]:
# Reporte detallado
print("REPORTE DE CLASIFICACION")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Activo', 'Churned']))

print("\nMATRIZ DE CONFUSION")
print("-" * 40)
cm = confusion_matrix(y_test, y_pred)
print(f"                  Predicho")
print(f"               Activo    Churned")
print(f"Real Activo     {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"Real Churned    {cm[1,0]:5d}    {cm[1,1]:5d}")

In [ ]:
# Feature importance
print("IMPORTANCIA DE FEATURES")
print("=" * 60)

feature_importance = pd.DataFrame({
    'feature': feature_cols_full,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 features mas importantes para predecir churn:")
for i, row in feature_importance.head(5).iterrows():
    bar = '#' * int(row['importance'] * 50)
    print(f"  {row['feature']:<20} {row['importance']:.4f} {bar}")

## 9. Predicciones y Estrategias de Retencion

In [ ]:
# Crear nuevos clientes para evaluacion
new_customers = pd.DataFrame({
    'customer_id': ['NEW-001', 'NEW-002', 'NEW-003', 'NEW-004', 'NEW-005', 'NEW-006'],
    'tenure_months': [24, 3, 48, 6, 36, 2],
    'recency_days': [15, 120, 7, 180, 30, 90],
    'frequency': [25, 3, 45, 5, 20, 2],
    'monetary_value': [2500.00, 150.00, 5000.00, 300.00, 1800.00, 80.00],
    'avg_order_value': [100.00, 50.00, 111.11, 60.00, 90.00, 40.00],
    'num_returns': [1, 4, 0, 3, 2, 5],
    'num_complaints': [0, 3, 0, 2, 1, 4],
    'channel_enc': [0, 1, 2, 1, 0, 1],  # online, store, both
    'loyalty_program': [1, 0, 1, 0, 1, 0],
    'marketing_opt_in': [1, 0, 1, 0, 1, 0],
    'description': [
        'Cliente leal con compras regulares',
        'Cliente nuevo inactivo con quejas',
        'Champion - alto valor y frecuencia',
        'Cliente inactivo por 6 meses',
        'Cliente estable con programa fidelidad',
        'Cliente nuevo con multiples devoluciones'
    ]
})

print("EVALUACION DE RIESGO DE CHURN")
print("=" * 90)

# Preparar y predecir
X_new = new_customers[feature_cols_full].values
X_new_scaled = scaler.transform(X_new)

predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)[:, 1]

print(f"{'ID':<10} {'Tenure':>8} {'Recency':>9} {'Freq':>6} {'Value':>10} {'Riesgo':>8} {'Estado':>10}")
print("-" * 90)

for i, row in new_customers.iterrows():
    cust_id = row['customer_id']
    tenure = row['tenure_months']
    recency = row['recency_days']
    freq = row['frequency']
    value = row['monetary_value']
    risk = probabilities[i] * 100
    status = "RIESGO" if predictions[i] == 1 else "Activo"
    flag = "" if predictions[i] == 0 else "!!!"
    
    print(f"{cust_id:<10} {tenure:>6}m {recency:>7}d {freq:>6} ${value:>9,.2f} {risk:>7.1f}% {status:>10} {flag}")

print("\n" + "-" * 90)
at_risk = predictions.sum()
at_risk_value = new_customers.loc[predictions == 1, 'monetary_value'].sum()
print(f"Clientes en riesgo de churn: {at_risk}")
print(f"Valor en riesgo: ${at_risk_value:,.2f}")

In [ ]:
# Estrategias de retencion personalizadas
print("\nESTRATEGIAS DE RETENCION PERSONALIZADAS")
print("=" * 60)

def get_retention_strategy(row, risk_prob):
    """Genera estrategia de retencion basada en perfil del cliente."""
    strategies = []
    
    if risk_prob > 0.7:
        strategies.append("URGENTE: Contacto personal del gerente")
    
    if row['recency_days'] > 60:
        strategies.append("Campana de reactivacion con descuento 20%")
    
    if row['num_complaints'] > 2:
        strategies.append("Resolver quejas pendientes + compensacion")
    
    if row['num_returns'] > 3:
        strategies.append("Revisar calidad de productos comprados")
    
    if row['loyalty_program'] == 0:
        strategies.append("Invitar a programa de fidelidad con bonus")
    
    if row['marketing_opt_in'] == 0:
        strategies.append("Ofrecer incentivo por suscripcion a newsletter")
    
    if not strategies:
        strategies.append("Mantener engagement actual")
    
    return strategies

for i, row in new_customers.iterrows():
    if predictions[i] == 1:
        print(f"\n{row['customer_id']}: {row['description']}")
        print("-" * 40)
        print(f"   Probabilidad de churn: {probabilities[i]*100:.1f}%")
        print(f"   Valor del cliente: ${row['monetary_value']:,.2f}")
        print("   Estrategias recomendadas:")
        for strategy in get_retention_strategy(row, probabilities[i]):
            print(f"      {strategy}")

## 10. Resumen y Garantias de Privacidad

In [ ]:
print("RESUMEN DEL DEMO RETAIL")
print("=" * 60)

print("""
GARANTIAS DE PRIVACIDAD
-----------------------
 Los datos de cada retailer NUNCA se comparten en plaintext
 Toda la informacion de clientes esta encriptada (CKKS 128-bit)
 El modelo se entrena sobre datos encriptados
 Solo el retailer puede ver sus propios clientes
 Votacion commit-reveal evita manipulacion
 Audit trail completo en Arbitrum blockchain

METRICAS DEL CONSORCIO
----------------------
""")
print(f"  Retailers participantes: {len(RETAIL_CONFIG['retailers'])}")
print(f"  Total clientes: {RETAIL_CONFIG['n_samples']:,}")
print(f"  Clientes en riesgo: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"  Accuracy del modelo: {accuracy*100:.2f}%")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  Revenue total: ${df_customers['monetary_value'].sum():,.2f}")

# Calcular potencial de ahorro
avg_customer_value = df_customers['monetary_value'].mean()
potential_saves = int(y.sum() * 0.3)  # Asumimos 30% de retencion
potential_revenue = potential_saves * avg_customer_value

print(f"\nIMPACTO POTENCIAL")
print("-" * 40)
print(f"  Clientes que podriamos retener: ~{potential_saves}")
print(f"  Revenue potencial a salvar: ~${potential_revenue:,.2f}")

print("""
CUMPLIMIENTO REGULATORIO
------------------------
 Ley de Proteccion de Datos Personales 25.326 (Argentina)
 Ley de Defensa del Consumidor
 GDPR (Operaciones internacionales)

SMART CONTRACTS DESPLEGADOS
---------------------------
Red: Arbitrum Sepolia (Testnet)
""")
print("  Governance:    0xda52326d106A91A1F22A0c41Be2dc1F531C01F11")
print("  Registry:      0x1296cCeF7803Bff51FB690afCFc586E7012417b8")
print("  Verifier:      0xa5f04E0aefe55173C91b949Aa2385f0228dd2921")
print("\nExplorador: https://sepolia.arbiscan.io")

---

## Proximos Pasos

1. **Probar con datos reales**: Conectar con la API en produccion
2. **Unirse a un consorcio existente**: Dashboard > Consorcios > Buscar
3. **Explorar otros verticales**: Fintech, Healthcare, Insurance, Government

**Documentacion**: https://apifhe.xcapit.com/api/v2/docs/